# Módulo 00 — escolher o modelo do curso: um protocolo, não um palpite

O [modelo do curso](srag_model_walkthrough.ipynb) é um XGBoost com cinco
hiperparâmetros. **Este caderno é a auditoria de como esses cinco números
foram escolhidos** — e a auditoria só vale porque o critério foi escrito
ANTES de qualquer leitura do teste, com hipótese do autor e consequência
declarada por extenso (§1).

O estudo inteiro roda em `tools/srag_selection.py --search`; a evidência
vive em [`gold/selection_metrics.json`](../gold/selection_metrics.json) e
o card gerado em [`SELECTION.md`](../SELECTION.md). Este caderno
**reproduz ao vivo as partes baratas** e confere cada número contra esse
registro — reprodução, não recontagem da narrativa. As buscas completas e
os braços de medição estão no
[internals](srag_selection_internals.ipynb).

O que ele deliberadamente NÃO faz: re-sincronizar os módulos de método. A
adoção do vencedor em `srag_model.XGB_PARAMS` já aconteceu; re-executar os
cinco módulos já escritos (01–05) sobre o modelo novo é PR posterior, e
fica registrado como pendência no §9 — não como pendura.

In [1]:
%pip install -q pandas pyarrow scikit-learn xgboost
# versões fixadas em requirements.txt; outras versões movem a terceira casa

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import pathlib
import subprocess
import sys

import pandas as pd

pd.set_option("display.width", 120)

REPO = pathlib.Path.cwd()
while REPO != REPO.parent and not (REPO / "tools" / "srag_model.py").exists():
    REPO = REPO.parent
if not (REPO / "tools" / "srag_model.py").exists():
    import tempfile
    REPO = pathlib.Path(tempfile.gettempdir()) / "interpretable-ml-lectures"
    if not REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/wbendinelli/interpretable-ml-lectures.git",
                        str(REPO)], check=True)
sys.path.insert(0, str(REPO / "tools"))
import srag_model as M
import srag_selection as S

RANDOM_STATE = 42
# A folga das conferências. O ci.yml declara que BLAS e plataforma movem a
# terceira casa entre um runner Linux e a caixa macOS que produziu o
# registro; as células imprimem a igualdade EXATA em 4 casas ao lado, que
# é o que se espera aqui, e o assert só reprova divergência REAL.
TOL = 1e-3

g = M.load_gold(REPO / "modules/00-dataset/gold/gold_covid_obito_sample.parquet")
SM = json.loads(
    (REPO / "modules/00-dataset/gold/selection_metrics.json").read_text(encoding="utf-8")
)


def confere(rotulo, vivo, registro):
    """Medida viva x registro commitado, chave a chave: imprime e reprova."""
    d = pd.DataFrame([{"chave": k,
                       "vivo": round(float(vivo[k]), 4),
                       "registro": round(float(registro[k]), 4)} for k in registro])
    d["delta"] = (d["vivo"] - d["registro"]).round(4)
    iguais = int((d["vivo"] == d["registro"]).sum())
    print(f"{rotulo}: {iguais}/{len(d)} idênticos em 4 casas,"
          f" |delta| máximo {d['delta'].abs().max():.4f}")
    assert (d["delta"].abs() <= TOL).all(), f"{rotulo} divergiu do registro além de {TOL}"


print(f"amostra: {len(g):,} linhas x {len(g.columns)} colunas")
print(f"registro: {sum(len(v) for v in SM['buscas'].values())} configurações buscadas,"
      f" {len(SM['placar'])} linhas de placar")
print(f"vencedor registrado: {SM['escolha']['vencedor']}")

amostra: 240,290 linhas x 50 colunas
registro: 73 configurações buscadas, 12 linhas de placar
vencedor registrado: xgb


## 1. O protocolo, antes de qualquer número

Pré-registro é a diferença entre "o XGBoost venceu" e "eu escolhi o
XGBoost". O critério abaixo é uma constante do módulo — está no código,
datado, com a hipótese do autor e a consequência escritas antes da
primeira busca. Ler primeiro, medir depois.

In [3]:
print(S.CRITERIO)

Pré-registrado em 2026-09-01, antes de qualquer leitura do teste.

1. PRIMÁRIO — maior AUC-ROC na VALIDAÇÃO (2023 inteiro). O teste (2024)
   nunca entra na escolha.
2. EMPATE TÉCNICO — modelos dentro de 1 erro-padrão bootstrap pareado
   (200 reamostras, semente 42) da melhor AUC de validação estão empatados.
3. DESEMPATE 1 — calibração: menor |previsto médio − observado| na validação.
4. DESEMPATE 2 — parcimônia: menor número de parâmetros ajustados.
5. DESEMPATE 3 — determinância: sem subamostragem estocástica.
6. O teste é lido UMA vez, depois de escolhido, e não pode mudar a escolha.
   Qualquer leitura adicional é rotulada "exploratória".
7. HIPÓTESE DO AUTOR, registrada antes de medir: o XGBoost vence, por
   ≥ 0,02 de AUC sobre a logística e ≥ 0,01 sobre a floresta.
8. CONSEQUÊNCIA DECLARADA: se o vencedor tunado divergir de XGB_PARAMS,
   o modelo do curso muda; os módulos de método são re-sincronizados em
   PR posterior.


As três janelas em que ele opera. A seleção enxerga só a do meio; 2024 é
lido uma vez, no §7.

In [4]:
splits = (g.groupby("split", observed=True)["y_obito"]
          .agg(n="size", obitos="sum", prevalencia="mean")
          .reindex(["train", "val", "test"]))
splits["prevalencia"] = splits["prevalencia"].round(4)
splits["janela"] = ["<= 2022", "2023 (seleção)", "2024 (lido uma vez)"]
print(splits.to_string())

for chave, nome in (("treino", "train"), ("val", "val"), ("test", "test")):
    assert int(splits.loc[nome, "n"]) == SM["amostra"][chave]
print("\nos três tamanhos conferem com o registro do estudo")

            n  obitos  prevalencia               janela
split                                                  
train  200000   62719       0.3136              <= 2022
val     24148    4572       0.1893       2023 (seleção)
test    16142    2945       0.1824  2024 (lido uma vez)

os três tamanhos conferem com o registro do estudo


## 2. Por que não k-fold embaralhado: a armadilha, medida

O default do mercado é `RandomizedSearchCV(cv=5)`: cinco dobras
embaralhadas dentro do treino. Sobre esta base isso quer dizer que cada
dobra de validação tem, na dobra de treino, pacientes da mesma semana
epidemiológica — a pretensão i.i.d. debaixo de uma deriva de regime que o
módulo 00 mede desde o funil.

A objeção não é descartada por argumento. As mesmas quatro configurações,
com o mesmo orçamento — a única diferença entre elas é a profundidade —
foram julgadas pelos dois protocolos quando o estudo rodou (`--search`),
e a tabela completa vive no registro commitado. A célula abaixo reproduz
a mecânica AO VIVO numa configuração só — a favorita do k-fold — nos dois
protocolos, confere contra o registro, e lê o resto da tabela de lá:
demonstração continua sendo demonstração, sem pagar os 24 fits de novo em
cada execução.

In [5]:
# a favorita do k-fold, reproduzida ao vivo nos dois protocolos
uma = dict(S.TRAP_XGB, max_depth=[7])
kf1 = S.search_kfold("xgb", g, uma)              # 5 dobras embaralhadas no treino
tp1 = S.search("xgb", g, n_iter=1, espaco=uma)   # a validação temporal do estudo

reg_k = {r["config"].replace('"', ""): r["auc_kfold"]
         for r in SM["kfold_trap"]["kfold"]}
reg_t = {r["config"].replace('"', ""): r["auc"]
         for r in SM["kfold_trap"]["temporal"]}
viva = kf1.iloc[0]["config"].replace('"', "")
confere("k-fold ao vivo (max_depth=7)",
        {viva: kf1.iloc[0]["auc_kfold"]}, {viva: reg_k[viva]})
confere("temporal ao vivo (max_depth=7)",
        {viva: tp1.iloc[0]["auc"]}, {viva: reg_t[viva]})

# a tabela completa, do registro commitado (--search a mediu uma vez)
armadilha = pd.DataFrame(
    {"config": list(reg_k), "auc_kfold": list(reg_k.values())})
armadilha["auc_val_2023"] = armadilha["config"].map(reg_t)
armadilha["posto_kfold"] = (armadilha["auc_kfold"]
                            .rank(ascending=False, method="min").astype(int))
armadilha["posto_temporal"] = (armadilha["auc_val_2023"]
                               .rank(ascending=False, method="min").astype(int))
armadilha["inflacao"] = (armadilha["auc_kfold"]
                         - armadilha["auc_val_2023"]).round(4)
armadilha = armadilha.sort_values("posto_kfold").reset_index(drop=True)
print("\n" + armadilha[["config", "auc_kfold", "posto_kfold", "auc_val_2023",
                        "posto_temporal", "inflacao"]].to_string(index=False))

k-fold ao vivo (max_depth=7): 1/1 idênticos em 4 casas, |delta| máximo 0.0000
temporal ao vivo (max_depth=7): 1/1 idênticos em 4 casas, |delta| máximo 0.0000

                                                config  auc_kfold  posto_kfold  auc_val_2023  posto_temporal  inflacao
{learning_rate: 0.08, max_depth: 7, n_estimators: 150}     0.7573            1        0.7504               2    0.0069
{learning_rate: 0.08, max_depth: 5, n_estimators: 150}     0.7558            2        0.7520               1    0.0038
{learning_rate: 0.08, max_depth: 9, n_estimators: 150}     0.7555            3        0.7441               3    0.0114
{learning_rate: 0.08, max_depth: 3, n_estimators: 150}     0.7491            4        0.7434               4    0.0057


In [6]:
vence_kfold = armadilha.sort_values(["posto_kfold", "config"]).iloc[0]["config"]
vence_temporal = (armadilha.sort_values(["posto_temporal", "config"])
                  .iloc[0]["config"])
print(f"5-fold embaralhado escolheria: {vence_kfold}")
print(f"validação temporal escolhe:   {vence_temporal}")
print(f"as duas concordam? {vence_kfold == vence_temporal}")

5-fold embaralhado escolheria: {learning_rate: 0.08, max_depth: 7, n_estimators: 150}
validação temporal escolhe:   {learning_rate: 0.08, max_depth: 5, n_estimators: 150}
as duas concordam? False


A tabela é a seção inteira, em três leituras:

1. a coluna `inflacao` é positiva nas quatro configurações — a AUC
   embaralhada é maior que a temporal na MESMA configuração, e não porque
   o modelo tenha melhorado: a pergunta ficou mais fácil. Prever um
   paciente de 2021 tendo visto vizinhos dele na dobra de treino não é a
   pergunta que o curso faz;
2. as duas ordenações não coincidem no topo — a célula imprime as duas
   escolhas lado a lado. O protocolo default trocaria o modelo do curso;
3. a troca seria invisível. Não há nada de errado na cara da saída do
   `cv=5`: é uma média de cinco dobras, bem-comportada, e nada nela avisa
   que as dobras dividem o regime entre si. É exatamente isso que a torna
   perigosa.

A validação temporal também tem um custo, e ele está declarado: ela julga
com um ano só (2023), então o EP do §5 é maior do que seria com cinco
dobras. O estudo prefere um erro-padrão honesto a uma média enviesada.

## 3. O zoo: seis modelos, três matrizes

Seis formas funcionais, do constante ao boosting. A coluna que pesa tanto
quanto a AUC é `forma`: é ela que os módulos de método terão de explicar
depois — um objeto que ninguém consegue explicar não serve a este curso,
mesmo que ganhe. `orçamento` é o `N_ITER` declarado antes de olhar, o que
impede a busca de virar "mais trials até o número ficar bonito".

In [7]:
mats = S.matrices(g)
zoo = pd.DataFrame([{"modelo": n,
                     "Molnar": S.MOLNAR[n],
                     "forma": S.FORMA[n],
                     "matriz": S.MATRIZ_DE[n],
                     "colunas": int(mats[S.MATRIZ_DE[n]].shape[1]),
                     "orçamento": S.N_ITER[n],
                     "buscadas": len(SM["buscas"][n])} for n in S.MODELOS])
print(zoo.to_string(index=False))
print("\ncolunas por matriz:", {k: int(v.shape[1]) for k, v in mats.items()})
print("NaN na matriz linear:", int(mats["linear"].isna().sum().sum()))

  modelo Molnar                         forma matriz  colunas  orçamento  buscadas
   dummy      —                     constante linear       91          1         1
     lpm cap. 6           linear aditivo em p linear       91          1         1
   logit cap. 7    linear aditivo em log-odds linear       91          7         7
  arvore cap. 9           regras hierárquicas arvore      124         24        24
floresta      —          ensemble por bagging arvore      124         10        10
     xgb      — ensemble aditivo por boosting    xgb       40         30        30

colunas por matriz: {'xgb': 40, 'linear': 91, 'arvore': 124}
NaN na matriz linear: 0


O `SimpleImputer(strategy="median")` do pipeline linear é um **no-op**
nesta base — a célula acima imprime quantos NaN a matriz linear tem. Ele
fica por duas razões, e nenhuma é inércia: é o MESMO pré-processamento de
`srag_model.fit_models`, e a logística tunada aqui só é comparável linha
a linha com a do modelo do curso se a única diferença for o
hiperparâmetro; e a ausência hoje é nível explícito (`9-Ignorado`,
`desconhecido`) por decisão do Ouro, uma decisão que pode mudar — no dia
que mudar, o passo já está no lugar e ninguém precisa lembrar dele.

### Fora do estudo, de propósito

- **GAM (Molnar cap. 8)** — a idade tem um U que a reta achata; escolher
  os termos suaves É a lição do capítulo, e ela não cabe numa linha de
  placar.
- **Regras de decisão (cap. 10)** — SE-ENTÃO sobre checkboxes é
  exatamente o que o funil torna traiçoeiro; a árvore já entrega aqui a
  versão hierárquica da mesma ideia.
- **RuleFit (cap. 11)** — gera regras e depois faz L1 sobre elas: dois
  capítulos empilhados, com subamostra própria.

Nenhum dos três está fora por qualidade. Cada um vira módulo próprio —
ver [ROADMAP.md](../../../ROADMAP.md).

## 4. Cada um nos seus melhores parâmetros

`MELHORES` é o bloco que a busca imprimiu e que foi colado de volta no
módulo (`--check-card` reprova se ele divergir do JSON). Aqui os seis são
reajustados no treino, cada um nos seus melhores parâmetros: é este
conjunto de seis que o placar do §5 mede.

In [8]:
modelos, tempos = {}, {}
for nome in S.MODELOS:
    modelos[nome], tempos[nome] = S.fit_one(nome, g, S.MELHORES[nome])

print(pd.DataFrame([{"modelo": n,
                     "melhores parâmetros": json.dumps(S.MELHORES[n], sort_keys=True),
                     "fit (s)": round(tempos[n], 1)} for n in S.MODELOS]
                   ).to_string(index=False))
assert S.MELHORES == SM["escolha"]["melhores"], "MELHORES divergiu do JSON"
print("\nMELHORES confere com escolha.melhores do registro")

  modelo                                                                                      melhores parâmetros  fit (s)
   dummy                                                                                    {"strategy": "prior"}      0.0
     lpm                                                                                                       {}      1.0
   logit                                                                                             {"C": 0.003}      0.7
  arvore                  {"ccp_alpha": 0.0, "criterion": "log_loss", "max_depth": null, "min_samples_leaf": 200}      1.4
floresta                                         {"max_depth": 16, "max_features": "sqrt", "min_samples_leaf": 5}      7.5
     xgb {"learning_rate": 0.05, "max_depth": 4, "min_child_weight": 100, "n_estimators": 800, "reg_lambda": 5.0}      2.4

MELHORES confere com escolha.melhores do registro


O LPM não tem hiperparâmetro nenhum — é o ponto do capítulo 6. O que ele
tem é uma patologia, e ela é o produto da linha: a célula abaixo imprime
quantas previsões dele caem FORA de [0,1] na validação, e o intervalo
bruto antes do clip que `predict` aplica.

In [9]:
clip = S.clip_report("lpm", modelos["lpm"], g, "val")
n_val = int((g["split"] == "val").sum())
print(f"LPM na validação: {clip['fora_01']:,} previsões fora de [0,1]"
      f" ({100 * clip['fora_01'] / n_val:.1f}% do split)")
print(f"intervalo bruto: [{clip['min']}, {clip['max']}]")
confere("clip do LPM", clip,
        {k: v for k, v in next(r for r in SM["lpm_clip"] if r["split"] == "val").items()
         if k != "split"})

LPM na validação: 4,999 previsões fora de [0,1] (20.7% do split)
intervalo bruto: [-0.4592, 1.2987]
clip do LPM: 3/3 idênticos em 4 casas, |delta| máximo 0.0000


Uma probabilidade negativa não é um bug de implementação: é a reta
respondendo o que a reta responde. O clip é um curativo, e a fração
impressa acima é o tamanho do ferimento — com o extremo bruto passando de
1 no outro lado. É por isso que o LPM está no zoo: ele perde por uma
razão que se vê, não por uma décima de AUC.

## 5. O placar de VALIDAÇÃO

Uma linha por modelo, medida em 2023. `n_parametros` é a coluna da regra
4 (parcimônia) e `ece` a da regra 3 (calibração): o critério precisa das
duas ANTES de saber quem ganhou, e é por isso que elas estão no placar em
vez de aparecerem depois, quando já dariam para escolher o desempate que
favorece o resultado desejado.

In [10]:
placar_val = S.leaderboard(modelos, tempos, g, ("val",))
mostrar = ["modelo", "n_features", "n_parametros", "auc", "pr_auc", "lift_pr",
           "brier", "logloss", "ece", "previsto_medio", "observado", "fit_s"]
print(placar_val[mostrar].to_string(index=False))

reg_val = {r["modelo"]: r["auc"] for r in SM["placar"] if r["split"] == "val"}
vivo_val = dict(zip(placar_val["modelo"], placar_val["auc"]))
confere("AUC de validação", vivo_val, reg_val)

  modelo  n_features  n_parametros    auc  pr_auc  lift_pr  brier  logloss    ece  previsto_medio  observado  fit_s
   dummy          91             1 0.5000  0.1893   1.0000 0.1689   0.5246 0.1243          0.3136     0.1893    0.0
     lpm          91            92 0.7232  0.3371   1.7805 0.1552   0.5091 0.1050          0.2823     0.1893    1.0
   logit          91            92 0.7214  0.3347   1.7680 0.1545   0.4646 0.0868          0.2762     0.1893    0.7
  arvore         124          1507 0.7041  0.3156   1.6668 0.1434   0.4481 0.0354          0.2247     0.1893    1.4
floresta         124       2736410 0.7403  0.3616   1.9101 0.1454   0.4563 0.0897          0.2790     0.1893    7.5
     xgb          40         20284 0.7564  0.3817   2.0159 0.1356   0.4209 0.0316          0.2209     0.1893    2.4
AUC de validação: 6/6 idênticos em 4 casas, |delta| máximo 0.0000


In [11]:
va = (g["split"] == "val").to_numpy()
y_va = g.loc[va, "y_obito"].to_numpy()
predicoes = {n: S.predict(n, modelos[n], g, va) for n in S.MODELOS}
ep = S.bootstrap_auc(y_va, predicoes)
print(ep.to_string(index=False))

confere("AUC média do bootstrap", dict(zip(ep["modelo"], ep["auc_media"])),
        {r["modelo"]: r["auc_media"] for r in SM["bootstrap"]})

  modelo  auc_media     ep  delta_lider  delta_ep  reamostras  lider
   dummy     0.5000 0.0000      -0.2564    0.0039         200  False
     lpm     0.7232 0.0038      -0.0332    0.0026         200  False
   logit     0.7214 0.0038      -0.0350    0.0027         200  False
  arvore     0.7039 0.0040      -0.0526    0.0034         200  False
floresta     0.7404 0.0039      -0.0160    0.0018         200  False
     xgb     0.7564 0.0039       0.0000    0.0000         200   True
AUC média do bootstrap: 6/6 idênticos em 4 casas, |delta| máximo 0.0000


Três leituras que a AUC sozinha não dá:

- o `dummy` não é piada. Ele é a prevalência, a linha de base do `lift_pr`
  e a resposta medida a "por que não usar `class_weight`": o desbalanço
  entra no placar como uma linha, não como um truque de amostragem que
  destruiria a história de calibração (MANIFEST §2.7);
- `lpm` e `logit` ficam a um fio de distância na AUC — e o LPM chega a
  ficar na frente. O elo logístico não compra ranking: compra `ece` e,
  sobretudo, `logloss`. É o capítulo 6 contra o 7 em duas linhas de
  tabela, e a diferença aparece onde a probabilidade importa, não onde a
  ordem importa;
- o bootstrap é **pareado**: as mesmas 200 reamostras para todos. O que a
  regra 2 usa é `delta_ep`, não `ep` — dois modelos que erram nos mesmos
  pacientes têm diferença muito mais estável do que os intervalos
  individuais sugerem, e é a diferença que decide um empate técnico.

## 6. A escolha, feita por função

`pick_model` aplica o critério na ordem escrita e devolve o log das
regras que precisou usar. A escolha não é uma frase minha no fim do
caderno: é o retorno de uma função que recebe o placar de validação e o
bootstrap, e nada mais.

In [12]:
vencedor, log = S.pick_model(placar_val, ep)
print(f"vencedor: {vencedor}")
for linha in log:
    print(f"  - {linha}")

assert vencedor == SM["escolha"]["vencedor"], "o vencedor vivo divergiu do registrado"
print(f"\nmesmo vencedor do registro: {vencedor}")
print(f"mesmo log, palavra por palavra? {log == SM['escolha']['log']}")

vencedor: xgb
  - regra 1: xgb lidera a AUC de validação (0,7564)
  - regra 2: nenhum modelo dentro de 1 EP (0,0018) do líder — sem empate

mesmo vencedor do registro: xgb
mesmo log, palavra por palavra? True


E a guarda, que é o ponto do módulo: a função levanta exceção se receber
qualquer linha de `split == "test"`. Não é decoração — é a única forma de
o "o teste não entra na escolha" ser uma garantia de código em vez de uma
promessa de quem escreveu.

In [13]:
contrabando = placar_val.head(1).copy()
contrabando["split"] = "test"          # uma linha de 2024, contrabandeada
tentativa = pd.concat([placar_val, contrabando], ignore_index=True)
try:
    S.pick_model(tentativa, ep)
except ValueError as erro:
    print("a função se recusa — o teste não entra na escolha:")
    print(f"  ValueError: {erro}")
else:
    raise AssertionError("a guarda de pick_model não disparou — isto seria um bug")

a função se recusa — o teste não entra na escolha:
  ValueError: pick_model recebeu linhas de split=test: a escolha nunca vê 2024


## 7. O teste (2024), lido uma vez

Os seis modelos já estão ajustados desde o §4. Nada é reajustado para
chegar até aqui: o placar simplesmente passa a incluir o split de 2024,
DEPOIS de a escolha estar feita e registrada acima.

In [14]:
placar = S.leaderboard(modelos, tempos, g, ("val", "test"))
print(placar[placar["split"] == "test"][mostrar].to_string(index=False))

comp = placar.pivot(index="modelo", columns="split", values="auc").reindex(list(S.MODELOS))
comp.columns = ["AUC test", "AUC val"]
comp = comp[["AUC val", "AUC test"]]
comp["delta AUC"] = (comp["AUC test"] - comp["AUC val"]).round(4)
gap = (placar.assign(gap=(placar["previsto_medio"] - placar["observado"]).round(4))
       .pivot(index="modelo", columns="split", values="gap").reindex(list(S.MODELOS)))
gap.columns = ["gap test", "gap val"]
print("\nvalidação -> teste (gap = previsto médio - observado):")
print(pd.concat([comp, gap[["gap val", "gap test"]]], axis=1).to_string())

reg_test = {r["modelo"]: r["auc"] for r in SM["placar"] if r["split"] == "test"}
confere("AUC de teste", dict(comp["AUC test"]), reg_test)

  modelo  n_features  n_parametros    auc  pr_auc  lift_pr  brier  logloss    ece  previsto_medio  observado  fit_s
   dummy          91             1 0.5000  0.1824   1.0000 0.1664   0.5192 0.1312          0.3136     0.1824    0.0
     lpm          91            92 0.7260  0.3379   1.8519 0.1475   0.4978 0.0908          0.2573     0.1824    1.0
   logit          91            92 0.7255  0.3363   1.8434 0.1449   0.4422 0.0674          0.2427     0.1824    0.7
  arvore         124          1507 0.7163  0.3128   1.7143 0.1386   0.4476 0.0385          0.2210     0.1824    1.4
floresta         124       2736410 0.7476  0.3698   2.0270 0.1412   0.4464 0.0919          0.2743     0.1824    7.5
     xgb          40         20284 0.7644  0.3888   2.1310 0.1308   0.4090 0.0329          0.2154     0.1824    2.4

validação -> teste (gap = previsto médio - observado):
          AUC val  AUC test  delta AUC  gap val  gap test
modelo                                                   
dummy      0.500

Duas leituras da tabela acima:

- a AUC sobe de 2023 para 2024 em todos os candidatos não triviais — a
  **discriminação viaja** entre regimes, o ranking de risco continua
  valendo num mundo que mudou;
- o gap de calibração é positivo nos dois anos para os seis: todo mundo
  prevê óbito demais, porque todo mundo aprendeu as letalidades de
  2020–22. Ele encolhe nos lineares e cresce nas árvores — a direção não
  é uniforme, e é por isso que a coluna está impressa em vez de resumida
  numa frase.

O vencedor da validação continua vencedor no teste. Isso é confirmação,
não prova: a regra 6 vale nos dois sentidos, e se o teste tivesse
apontado outro modelo a escolha registrada no §6 continuaria sendo a
escolha — o que mudaria é o que o card teria de contar.

## 8. O que o mercado faria (e o que custaria aqui)

Cada linha desta tabela é uma decisão que alguém vai querer "corrigir"
neste repositório. Ela é uma constante do módulo (`S.MERCADO`) para que a
correção tenha de passar pelo motivo escrito, em vez de acontecer num
commit de arrumação.

In [15]:
mercado = pd.DataFrame(list(S.MERCADO),
                       columns=["prática", "o que custaria aqui", "o que este estudo faz"])
print(f"{len(mercado)} práticas de mercado, uma por linha de S.MERCADO:\n")
for i, r in mercado.iterrows():
    print(f"{i + 1}. {r['prática']}")
    print(f"   custaria aqui : {r['o que custaria aqui']}")
    print(f"   este estudo   : {r['o que este estudo faz']}")

7 práticas de mercado, uma por linha de S.MERCADO:

1. `RandomizedSearchCV(cv=5` embaralhado`)`
   custaria aqui : mistura 2020–22 nas dobras
   este estudo   : seleção na val 2023 (a armadilha, medida acima)
2. Optuna, 100–500 trials
   custaria aqui : 500 leituras das mesmas 24k linhas = sobreajuste de seleção
   este estudo   : orçamento declarado + EP bootstrap
3. retreinar em treino+val antes do deploy
   custaria aqui : perderia o único conjunto de seleção honesto
   este estudo   : não retreina (o internals mede o que isso custa)
4. early stopping na validação
   custaria aqui : é busca de `n_estimators` fora do orçamento declarado
   este estudo   : `n_estimators` está na grade, contado
5. SMOTE / `class_weight`
   custaria aqui : destruiria a história de calibração (MANIFEST §2.7)
   este estudo   : a prevalência entra como linha do placar: o Dummy
6. espiar o teste "só uma vez" no meio
   custaria aqui : invalida a estimativa que o teste existe para dar
   este estudo   : `pi

Duas dessas linhas já foram **medidas** neste caderno, não argumentadas:
a do `cv=5` embaralhado é o §2, com as duas ordenações lado a lado; a de
espiar o teste é o §6, com a exceção que a `pick_model` levantou de
verdade. A linha do Optuna é a razão de o orçamento por modelo estar
declarado antes (§3): com o `ep` do §5 na mão, centenas de leituras da
MESMA validação escolheriam diferenças da ordem do erro-padrão — sobre-
ajuste de seleção, com cara de melhoria.

## 9. O que este estudo entrega

In [16]:
print(f"vencedor: {vencedor}")
print(f"hiperparâmetros: {json.dumps(S.MELHORES[vencedor], sort_keys=True)}\n")

print("adoção em tools/srag_model.py (regra 8 do critério):")
for k, v in sorted(S.MELHORES["xgb"].items()):
    print(f"  {k:<18} estudo {v!r:<6}   XGB_PARAMS {M.XGB_PARAMS.get(k)!r}")
assert set(S.MELHORES["xgb"]) <= set(M.XGB_PARAMS), "o modelo do curso não tem as chaves"
assert {k: M.XGB_PARAMS[k] for k in S.MELHORES["xgb"]} == S.MELHORES["xgb"]
print("  -> as cinco chaves do vencedor estão em XGB_PARAMS, com os mesmos valores")

print(f"\nXGB_PARAMS no dia da busca: "
      f"{json.dumps(SM['consequencia']['xgb_params_hoje'], sort_keys=True)}")
print(f"vencedor do estudo:         "
      f"{json.dumps(SM['consequencia']['xgb_params_vencedor'], sort_keys=True)}")
print(f"regra 8 acionada (divergiu): {SM['consequencia']['vencedor_diverge']}")

print("\npendência registrada: os cinco módulos de método já escritos (01-05)")
print("foram construídos sobre o modelo anterior e citam números dele; a")
print("re-sincronização é PR posterior.")

vencedor: xgb
hiperparâmetros: {"learning_rate": 0.05, "max_depth": 4, "min_child_weight": 100, "n_estimators": 800, "reg_lambda": 5.0}

adoção em tools/srag_model.py (regra 8 do critério):
  learning_rate      estudo 0.05     XGB_PARAMS 0.05
  max_depth          estudo 4        XGB_PARAMS 4
  min_child_weight   estudo 100      XGB_PARAMS 100
  n_estimators       estudo 800      XGB_PARAMS 800
  reg_lambda         estudo 5.0      XGB_PARAMS 5.0
  -> as cinco chaves do vencedor estão em XGB_PARAMS, com os mesmos valores

XGB_PARAMS no dia da busca: {"learning_rate": 0.08, "max_depth": 5, "n_estimators": 400}
vencedor do estudo:         {"learning_rate": 0.05, "max_depth": 4, "n_estimators": 800}
regra 8 acionada (divergiu): True

pendência registrada: os cinco módulos de método já escritos (01-05)
foram construídos sobre o modelo anterior e citam números dele; a
re-sincronização é PR posterior.


O que fica, e onde:

- [`SELECTION.md`](../SELECTION.md) — o card gerado, conferido por
  `--check-card` contra o JSON a cada rodada;
- [`gold/selection_metrics.json`](../gold/selection_metrics.json) — a
  evidência bruta: as seis buscas, as sondagens, o bootstrap, o placar;
- `S.CRITERIO`, `S.pick_model` — o protocolo como código, com a guarda do
  §6, para que a próxima pessoa que refizer o estudo não precise confiar
  em mim;
- a pendência do §9, que é dívida declarada e não conserto silencioso.

Ponteiros para o [internals](srag_selection_internals.ipynb): as buscas
completas e a reprodução de cada topo (§1, §2), one-hot contra ordinal
nas árvores (§3), o preço da determinância de semente (§4), a maldição do
vencedor com as leituras exploratórias rotuladas (§5), o que custaria
retreinar em treino+val (§6) e se o vencedor sobrevive à base cheia (§8).